# Chapter 24: Solving a Problem in an Interview

The programs from Chapter 24 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## One Problem, Three Solutions

`two_sum.py`

In [1]:
"""One problem, three solutions: find two positions whose numbers add
up to target. Return them as (i, j) with i < j, or None."""


def two_sum_brute(nums, target):
    """Try every pair: O(n^2) time, O(1) extra memory."""
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return i, j
    return None


def two_sum_sorted(nums, target):
    """Sort the positions by value, then move two pointers inward:
    O(n log n) time, O(n) memory."""
    order = sorted(range(len(nums)), key=nums.__getitem__)
    lo, hi = 0, len(order) - 1
    while lo < hi:
        total = nums[order[lo]] + nums[order[hi]]
        if total == target:
            return tuple(sorted((order[lo], order[hi])))
        if total < target:
            lo += 1
        else:
            hi -= 1
    return None


def two_sum_hash(nums, target):
    """Remember where each number was seen; look up its partner:
    O(n) time, O(n) memory."""
    seen = {}
    for j, x in enumerate(nums):
        if target - x in seen:
            return seen[target - x], j
        seen[x] = j
    return None

`bench_two_sum.py`

In [2]:
"""The three two-sum solutions, when no pair exists (the worst case)."""

import random

import bench
from two_sum import two_sum_brute, two_sum_hash, two_sum_sorted

rng = random.Random(2026)


def make(n):
    return [rng.randrange(0, 10**9, 2) for _ in range(n)], 1  # all even


sizes = [500, 1_000, 2_000, 4_000]
results = {
    "brute force": bench.doubling(two_sum_brute, make, sizes),
    "sort + pointers": bench.doubling(two_sum_sorted, make, sizes),
    "hash map": bench.doubling(two_sum_hash, make, sizes),
}
bench.table(results)
bench.chart(results, "ch24_two_sum.png", "Two sum: three solutions")

         n     brute force  sort + pointers        hash map
       500         5.06 ms          48.6 us         28.5 us
     1,000    19.6 ms x3.9      123 us x2.5    60.1 us x2.1
     2,000    82.8 ms x4.2      382 us x3.1     144 us x2.4
     4,000     343 ms x4.1      875 us x2.3     326 us x2.3


## Patterns and Their Signals

`signals.py`

In [3]:
"""Words in a problem statement that point to a pattern."""

SIGNALS = [
    ("sorted array, pair or triple with a sum", "two pointers"),
    (
        "contiguous subarray or substring, longest or shortest",
        "sliding window",
    ),
    ("count subarrays with a sum; seen before; pairs", "hash map"),
    ("sums of ranges, asked many times", "prefix sums"),
    ("fewest steps, shortest path, levels, unweighted", "BFS"),
    ("connected regions, all paths, explore everything", "DFS"),
    ("order with prerequisites", "topological sort"),
    ("k largest, k smallest, k closest, merge k", "heap"),
    ("minimum or maximum value that works, monotone", "binary search"),
    ("count ways, best total, choices that overlap", "dynamic programming"),
    ("all combinations, permutations, placements", "backtracking"),
    ("next greater or smaller element", "monotonic stack"),
]
for signal, pattern in SIGNALS:
    print(f"{pattern:>19}: {signal}")

       two pointers: sorted array, pair or triple with a sum
     sliding window: contiguous subarray or substring, longest or shortest
           hash map: count subarrays with a sum; seen before; pairs
        prefix sums: sums of ranges, asked many times
                BFS: fewest steps, shortest path, levels, unweighted
                DFS: connected regions, all paths, explore everything
   topological sort: order with prerequisites
               heap: k largest, k smallest, k closest, merge k
      binary search: minimum or maximum value that works, monotone
dynamic programming: count ways, best total, choices that overlap
       backtracking: all combinations, permutations, placements
    monotonic stack: next greater or smaller element


## Test Like a Professional: Stress Testing

`stress_test.py`

In [4]:
"""A stress test: compare a fast solution with a brute-force one on
thousands of small random inputs. The first difference is a bug, and
the input is small enough to debug by hand."""

import random

from two_sum import two_sum_brute, two_sum_hash, two_sum_sorted


def valid(answer, nums, target, brute_answer):
    """Is answer right? There may be several right pairs, so check the
    pair itself rather than comparing it with brute force's pair."""
    if answer is None:
        return brute_answer is None
    i, j = answer
    return 0 <= i < j < len(nums) and nums[i] + nums[j] == target


def stress(fast, slow, make_input, tries=5000, seed=2026):
    rng = random.Random(seed)
    for t in range(tries):
        nums, target = make_input(rng)
        answer = fast(nums, target)
        if not valid(answer, nums, target, slow(nums, target)):
            return f"FAILED on try {t}: {nums}, {target} -> {answer}"
    return f"passed {tries} random tests"


def small_case(rng):
    nums = [rng.randint(-5, 5) for _ in range(rng.randint(0, 6))]
    return nums, rng.randint(-6, 6)


def two_sum_buggy(nums, target):
    """Looks fine, but may use the same position twice."""
    seen = {x: i for i, x in enumerate(nums)}
    for i, x in enumerate(nums):
        if target - x in seen:
            return tuple(sorted((i, seen[target - x])))
    return None


print("hash:  ", stress(two_sum_hash, two_sum_brute, small_case))
print("sorted:", stress(two_sum_sorted, two_sum_brute, small_case))
print("buggy: ", stress(two_sum_buggy, two_sum_brute, small_case))

hash:   passed 5000 random tests
sorted: passed 5000 random tests
buggy:  FAILED on try 9: [-1, -1, 2, -3], 4 -> (2, 2)
